In [0]:
-- Databricks supports creating both regular views and materlized views

-- Regular View (https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-view)
CREATE OR REPLACE VIEW vw_monthly_project_spend_summary AS
SELECT
  `Project ID`,
  MONTH(`Month`) AS month,
  SUM(`Material Cost ($)`) AS total_material_cost,
  SUM(`Labor Cost ($)`) AS total_labor_cost,
  SUM(`Material Cost ($)` + `Labor Cost ($)`) AS total_spend
FROM actuals_sap
GROUP BY `Project ID`, MONTH(`Month`);

-- Materlized View (https://docs.databricks.com/aws/en/dlt/dbsql/materialized)
CREATE OR REPLACE MATERIALIZED VIEW mv_weekly_project_spend_summary
SCHEDULE CRON '0 7 * * * ? *' AT TIME ZONE 'America/Denver'
AS
SELECT
  `Project ID` as project_id,
  weekofyear(`Month`) AS week,
  YEAR(`Month`) AS year,
  SUM(`Material Cost ($)`) AS total_material_cost,
  SUM(`Labor Cost ($)`) AS total_labor_cost,
  SUM(`Material Cost ($)` + `Labor Cost ($)`) AS total_spend
FROM actuals_sap
GROUP BY `Project ID`, YEAR(`Month`), weekofyear(`Month`);

-- We can refresh a MV
REFRESH MATERIALIZED VIEW mv_weekly_project_spend_summary;
